# Output Validation

Check LLM outputs:
- Check outputs generated by the system before showing to users
- To ensure quality, relevance and safety of the responses provided to users
- Can use the Moderation API for outputs
- Use additional prompts to the model to evaluate output before display


In [ ]:
import os
os.environ['ACCESS_TOKEN_NAME'] = 'insert_access_token'

In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_ACCESS_TOKEN"],
)

In [4]:
def get_LLM_response(messages, model="zai-org/GLM-5.3:fireworks-ai", temperature=0):
    response = client.chat.completions.create(
        model= model,
        messages = messages,
        temperature = temperature,
    )

    return response.choices[0].message.content

### Output Validation Example

### _Moderation API for outputs_

Check LLM generated outputs using Moderation API before displaying to users.

If Moderation output indicates that the content is flagged, take appropriate action such as responding with another answer or generating a new response

In [6]:
final_response_to_customer = f"""
The SmartX ProPhone has a 6.1-inch display, 128GB storage, \
12MP dual camera, and 5G. The FotoSnap DSLR Camera \
has a 24.2MP sensor, 1080p video, 3-inch LCD, and \
interchangeable lenses. We have a variety of TVs, including \
the CineView 4K TV with a 55-inch display, 4K resolution, \
HDR, and smart TV features. We also have the SoundMax \
Home Theater system with 5.1 channel, 1000W output, wireless \
subwoofer, and Bluetooth. Do you have any specific questions \
about these products or any other products we offer?
"""

        response = client.Moderation.create(
            input=final_response_to_customer
        )
        moderation_output = response["results"][0]
        print(moderation_output)

### _Prompts to Evaluate output before display_
Ask model itself to evaluate the generated output and if it follows a certain rubric defined. This can be done by providing the generated ouput as input to the model to evaluate the quality of the output.

This mehtod is not used often as it will increase system costs but can be used if you want to evaluate immediately in certain cases.

Good Model Output Example:

Checks if agent responses to the customers question properly and responds with either Y or N


        system_message = f"""
        You are an assistant that evaluates whether \
        customer service agent responses sufficiently \
        answer customer questions, and also validates that \
        all the facts the assistant cites from the product \
        information are correct.
        The product information and user and customer \
        service agent messages will be delimited by \
        3 backticks, i.e. ```.
        Respond with a Y or N character, with no punctuation:
        Y - if the output sufficiently answers the question \
        AND the response correctly uses product information
        N - otherwise

        Output a single letter only.
        """

        customer_message = f"""
        tell me about the smartx pro phone and \
        the fotosnap camera, the dslr one. \
        Also tell me about your tvs"""

        product_information = """{ "name": "SmartX ProPhone", "category": "Smartphones and Accessories", "brand": "SmartX", "model_number": "SX-PP10", "warranty": "1 year", "rating": 4.6, "features": [ "6.1-inch display", "128GB storage", "12MP dual camera", "5G" ], "description": "A powerful smartphone with advanced camera features.", "price": 899.99 } { "name": "FotoSnap DSLR Camera", "category": "Cameras and Camcorders", "brand": "FotoSnap", "model_number": "FS-DSLR200", "warranty": "1 year", "rating": 4.7, "features": [ "24.2MP sensor", "1080p video", "3-inch LCD", "Interchangeable lenses" ], "description": "Capture stunning photos and videos with this versatile DSLR camera.", "price": 599.99 } { "name": "CineView 4K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-4K55", "warranty": "2 years", "rating": 4.8, "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], "description": "A stunning 4K TV with vibrant colors and smart features.", "price": 599.99 } { "name": "SoundMax Home Theater", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-HT100", "warranty": "1 year", "rating": 4.4, "features": [ "5.1 channel", "1000W output", "Wireless subwoofer", "Bluetooth" ], "description": "A powerful home theater system for an immersive audio experience.", "price": 399.99 } { "name": "CineView 8K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-8K65", "warranty": "2 years", "rating": 4.9, "features": [ "65-inch display", "8K resolution", "HDR", "Smart TV" ], "description": "Experience the future of television with this stunning 8K TV.", "price": 2999.99 } { "name": "SoundMax Soundbar", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-SB50", "warranty": "1 year", "rating": 4.3, "features": [ "2.1 channel", "300W output", "Wireless subwoofer", "Bluetooth" ], "description": "Upgrade your TV's audio with this sleek and powerful soundbar.", "price": 199.99 } { "name": "CineView OLED TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-OLED55", "warranty": "2 years", "rating": 4.7, "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], "description": "Experience true blacks and vibrant colors with this OLED TV.", "price": 1499.99 }"""

        
        q_a_pair = f"""
        Customer message: ```{customer_message}```
        Product information: ```{product_information}```
        Agent response: ```{final_response_to_customer}```

        Does the response use the retrieved information correctly?
        Does the response sufficiently answer the question

        Output Y or N
        """
        messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user', 'content': q_a_pair}
        ]

        response = get_LLM_response(messages)
        print(response)

Bad Model Output Example:

        another_response = "life is like a box of chocolates"
        q_a_pair = f"""
        Customer message: ```{customer_message}```
        Product information: ```{product_information}```
        Agent response: ```{another_response}```

        Does the response use the retrieved information correctly?
        Does the response sufficiently answer the question?

        Output Y or N
        """
        messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user', 'content': q_a_pair}
        ]

        response = get_LLM_response(messages)
        print(response)